# Proyecto final: Asociacion Vecinal San Pascual

Cristina Hernandez, Beatriz Jimenez, Macarena Vargas, Guillermo Ruiz

Caso real basado en la reunion con la AV San Pascual: 8 espacios fisicos
(Salas 1-7 sin la Sala 6, mas Gimnasio y Zona de Ocio) y un espacio virtual
`EXTERIOR` para actividades externas que no ocupan sala. El objetivo ya no es la
asistencia estimada sino el numero de sesiones/actividades realizadas. El numero
de sesiones de cada actividad es un TOPE semanal: nunca se supera, pero puede
quedarse corto si no hay forma de cuadrarlo.

In [1]:
import pyomo.environ as pe
import pyomo.opt as po
from pyomo.environ import NonNegativeReals, Binary

#### Create the model 

In [2]:
model = pe.ConcreteModel()

### Sets

* **$D$**: The set of days in the weekly horizon.
  $$D = \{\text{Monday, ..., Sunday}\}$$
* **$P$**: The set of daily shifts or periods.
  $$P = \{\text{Morning, Afternoon}\}$$
* **$H_p$**: The set of 30-min time slots available within each period $p \in P$.
  Morning $= \{9{:}00, ..., 12{:}30\}$, Afternoon $= \{16{:}00, ..., 20{:}30\}$.
* **$R$**: The set of rooms / spaces. The 8 real spaces used for activities plus a
  virtual `EXTERIOR` space for external activities (they do not occupy a real room).
* **$A$**: The set of community activities to be scheduled (real 2026 schedule).
* **$S$**: The set of monitors. One dedicated monitor per activity.
* **$\Omega$ (Spatio-temporal Space)**: $\Omega = \{(d, p, h) \mid d \in D, p \in P, h \in H_p\}$

In [3]:

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
PERIODS = ["Morning", "Afternoon"]

# 8 real spaces used for activities (Sala 6 is an unused office) + a virtual
# EXTERIOR space for external activities that do NOT occupy a physical room.
ROOMS = ["Sala1", "Sala2", "Sala3", "Sala4", "Sala5", "Sala7",
         "Gimnasio", "ZonaOcio", "EXTERIOR"]

# Real weekly activities taken from the 2026 schedule.
INTERNAL_ACTIVITIES = [
    "Yoga", "Pilates", "Gimnasia", "Estiramientos", "Hipopresivos",
    "Capoeira", "Baile", "DanzaVientre", "Teatro", "CoroMessaDiVoce", "CoroGospel",
    "Pintura", "Ingles", "InformaticaBasica", "InformaticaMedia",
    "UsoMovilBasico", "UsoMovilMedio", "MenteActiva", "EscrituraLectura",
    "Ortografia", "ClubLectura", "HistoriasDeVida", "AsocRafiki",
    "ReunionAV", "ReunionComunidad",
]
EXTERNAL_ACTIVITIES = ["Voleibol", "GrupoScout"]
ACTIVITIES = INTERNAL_ACTIVITIES + EXTERNAL_ACTIVITIES

# One dedicated monitor per activity. monitor_of maps each activity to its monitor.
SUPERVISORS = ["Mon_" + a for a in ACTIVITIES]
monitor_of = {a: "Mon_" + a for a in ACTIVITIES}

# Half-hour grid. Each index is a 30-min block; slot_label converts it to a real
# clock time. Morning 9:00-12:30, Afternoon 16:00-20:30 (13-16 closing gap).
HM = [1, 2, 3, 4, 5, 6, 7, 8]            # 9:00, 9:30, ... 12:30
HA = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]     # 16:00, 16:30, ... 20:30

def slot_label(p, h):
    base = 9 * 60 if p == "Morning" else 16 * 60
    minutes = base + (h - 1) * 30
    return "%02d:%02d" % (minutes // 60, minutes % 60)

# These sets allow the model to iterate over days, rooms, activities and monitors
model.d = pe.Set(initialize=DAYS, doc='Days of the week')
model.p = pe.Set(initialize=PERIODS, doc='Day periods (Morning/Afternoon)')
model.r = pe.Set(initialize=ROOMS, doc='Rooms / spaces (incl. virtual EXTERIOR)')
model.a = pe.Set(initialize=ACTIVITIES, doc='Types of activities')
model.s = pe.Set(initialize=SUPERVISORS, doc='Monitors (one dedicated per activity)')

# Defining the 30-min slots belonging to each period
model.Hm = pe.Set(initialize=HM)
model.Ha = pe.Set(initialize=HA)

# Initialization function to automatically assign slots based on the Period (p)
def H_init(m, p):
    return HM if p == "Morning" else HA

# model.H is an indexed set: model.H['Morning'] will contain [1,...,8]
model.H = pe.Set(model.p, initialize=H_init, doc='Available 30-min slots per period')

# 'dph' creates the "empty calendar" by combining (Day, Period, Slot)
model.dph = pe.Set(
    dimen=3,
    initialize=[(d, p, h) for d in DAYS for p in PERIODS for h in (HM if p=="Morning" else HA)],
    doc='Full spatio-temporal horizon of the model'
)

#### Parameters

* **$RA_{r,d,p,h}$**: Availability of room $r$ on day $d$, period $p$, slot $h$.
* **$T_a$**: Target (maximum) number of weekly sessions for activity $a$. Sessions
  never exceed $T_a$ but may fall short if they cannot be fitted in.
* **$Dur_a$**: Real duration of activity $a$ in consecutive 30-min blocks.
* **$NS_a$**: Supervision requirement for activity $a$.
* **$SA_{s,d,p,h}$**: Availability of monitor $s$ at slot $(d,p,h)$.
* **$Su_{a,r}$**: 1 if activity $a$ may take place in room $r$. Built from room
  *features* and activity *requirements*: a room is suitable iff it provides every
  feature the activity requires (so a special room can also host generic activities).
* **$Mus_a / Sil_a$**: 1 if activity $a$ is musical / requires silence.
* **$Adj_{r,r'}$**: 1 if rooms $r$ and $r'$ are physically adjacent.

In [4]:

# Room availability: physical rooms open Monday-Friday in both periods; closed at
# weekends. The virtual EXTERIOR space is always open (weekend volley / scouts).
WEEKEND = {"Saturday", "Sunday"}
room_avail = {
    (r, d, p, h): (1 if r == "EXTERIOR" else (0 if d in WEEKEND else 1))
    for r in ROOMS for d, p, h in model.dph
}
model.room_avail = pe.Param(
    model.r, model.dph,
    initialize=lambda m, r, d, p, h: room_avail[(r, d, p, h)],
    within=pe.Binary,
    doc='Binary parameter for room open/closed status'
)

# Upper bound of how many sessions we wuld like to have in any activity
target_sessions = {
    "Yoga": 5, "Pilates": 3, "Gimnasia": 3, "Estiramientos": 1, "Hipopresivos": 1,
    "Capoeira": 1, "Baile": 2, "DanzaVientre": 1, "Teatro": 1,
    "CoroMessaDiVoce": 1, "CoroGospel": 1, "Pintura": 1, "Ingles": 1,
    "InformaticaBasica": 1, "InformaticaMedia": 1, "UsoMovilBasico": 1,
    "UsoMovilMedio": 1, "MenteActiva": 2, "EscrituraLectura": 1, "Ortografia": 1,
    "ClubLectura": 1, "HistoriasDeVida": 2, "AsocRafiki": 1, "ReunionAV": 1,
    "ReunionComunidad": 2, "Voleibol": 2, "GrupoScout": 1,
}
model.target_sessions = pe.Param(
    model.a,
    initialize=lambda m, a: target_sessions[a],
    within=pe.NonNegativeIntegers,
    doc='Maximum (target) weekly frequency per activity'
)

# Real activity duration in consecutive 30-min blocks (2 blocks = 1 hour for example)
dur = {
    "Yoga": 2, "Pilates": 2, "Gimnasia": 2, "Estiramientos": 2, "Hipopresivos": 2,
    "Capoeira": 2, "Baile": 2, "DanzaVientre": 2, "Teatro": 3,
    "CoroMessaDiVoce": 3, "CoroGospel": 3, "Pintura": 4, "Ingles": 2,
    "InformaticaBasica": 2, "InformaticaMedia": 2, "UsoMovilBasico": 2,
    "UsoMovilMedio": 2, "MenteActiva": 2, "EscrituraLectura": 3, "Ortografia": 2,
    "ClubLectura": 3, "HistoriasDeVida": 3, "AsocRafiki": 2, "ReunionAV": 4,
    "ReunionComunidad": 4, "Voleibol": 4, "GrupoScout": 6,
}
model.dur = pe.Param(
    model.a,
    initialize=lambda m, a: dur[a],
    within=pe.PositiveIntegers,
    doc='Required consecutive 30-min blocks per session'
)

# Supervision flag: internal activities need their monitor; external activities are
# run by outside organisations and need no monitor
# this is an examplo constraint as we dont really know what activities need monitors
needs_sup = {a: (0 if a in EXTERNAL_ACTIVITIES else 1) for a in ACTIVITIES}
model.needs_sup = pe.Param(
    model.a,
    initialize=lambda m, a: needs_sup[a],
    within=pe.Binary,
    doc='Mandatory supervision flag'
)

# Monitor Availability Profile
# Simulation: monitors are unavailable on Sunday afternoons for rest periods
sup_avail = {
    (s, d, p, h): 0 if (d == "Sunday" and p == "Afternoon") else 1
    for s in SUPERVISORS
    for d, p, h in model.dph
}
model.sup_avail = pe.Param(
    model.s, model.dph,
    initialize=lambda m, s, d, p, h: sup_avail[(s, d, p, h)],
    within=pe.Binary,
    doc='Monitor availability constraints'
)

# Room suitability built from FEATURES vs REQUIREMENTS.
# Each room offers a set of features; each activity requires a set of features.
# A room is suitable for an activity iff it provides every required feature. Thus a
# room with mirrors (e.g. Sala2/Sala4) can host dance AND any generic activity,
# while quiet/concentration activities are restricted to the quiet room only.
room_features = {
    "Sala1": {"classroom"},
    "Sala2": {"mirror", "gym", "big", "classroom"},   # Sala Multiusos: mirrors + large
    "Sala3": {"quiet", "classroom"},                  # Biblioteca: the quiet room
    "Sala4": {"mirror", "classroom"},                 # Aula Taller: mirrors
    "Sala5": {"classroom"},
    "Sala7": {"classroom"},
    "Gimnasio": {"gym", "big"},                        # open sports floor
    "ZonaOcio": {"big", "gym", "classroom"},          # large multipurpose room
    "EXTERIOR": {"exterior"},
}
activity_req = {
    "Baile": {"mirror"}, "DanzaVientre": {"mirror"},
    "Yoga": {"gym"}, "Pilates": {"gym"}, "Gimnasia": {"gym"},
    "Estiramientos": {"gym"}, "Hipopresivos": {"gym"}, "Capoeira": {"gym"},
    "ClubLectura": {"quiet"}, "EscrituraLectura": {"quiet"},
    "Ortografia": {"quiet"}, "MenteActiva": {"quiet"},
    "Teatro": {"big"}, "CoroMessaDiVoce": {"big"}, "CoroGospel": {"big"},
    "Pintura": {"classroom"}, "Ingles": {"classroom"},
    "InformaticaBasica": {"classroom"}, "InformaticaMedia": {"classroom"},
    "UsoMovilBasico": {"classroom"}, "UsoMovilMedio": {"classroom"},
    "HistoriasDeVida": {"classroom"}, "AsocRafiki": {"classroom"},
    "ReunionAV": {"classroom"}, "ReunionComunidad": {"classroom"},
    "Voleibol": {"exterior"}, "GrupoScout": {"exterior"},
}
suitable = {(a, r): (1 if activity_req[a].issubset(room_features[r]) else 0)
            for a in ACTIVITIES for r in ROOMS}
model.suitable = pe.Param(
    model.a, model.r,
    initialize=lambda m, a, r: suitable[(a, r)],
    within=pe.Binary,
    doc='Activity-room suitability (requirements subset of room features)'
)

# Acoustic flags for the noise / adjacency constraint
MUSIC = {"Baile", "DanzaVientre", "Capoeira", "Teatro", "CoroMessaDiVoce", "CoroGospel"}
SILENCE = {"ClubLectura", "EscrituraLectura", "Ortografia", "MenteActiva"}
model.is_music = pe.Param(model.a, initialize=lambda m, a: 1 if a in MUSIC else 0,
                          within=pe.Binary, doc='Noisy/musical activity flag')
model.is_silence = pe.Param(model.a, initialize=lambda m, a: 1 if a in SILENCE else 0,
                            within=pe.Binary, doc='Silence-requiring activity flag')

# Physically adjacent rooms (from the floor plan). Symmetric pairs.
ADJ_PAIRS = [
    ("Sala1", "Sala3"), ("Sala3", "Sala4"), ("Sala1", "ZonaOcio"),
    ("Sala2", "Sala1"), ("Sala2", "ZonaOcio"),
    ("Gimnasio", "Sala5"), ("Gimnasio", "Sala7"), ("Sala5", "Sala7"),
]
adjacent = {(r1, r2): 0 for r1 in ROOMS for r2 in ROOMS}
for (r1, r2) in ADJ_PAIRS:
    adjacent[(r1, r2)] = 1
    adjacent[(r2, r1)] = 1
model.adjacent = pe.Param(
    model.r, model.r,
    initialize=lambda m, r1, r2: adjacent[(r1, r2)],
    within=pe.Binary,
    doc='Room adjacency matrix'
)

### Variables
* $X_{a,r,d,p,h} \in \{0, 1\}$: 1 if activity $a$ starts in room $r$ on day $d$,
  period $p$, slot $h$.
* $y_{a,r,d,p,h} \in \{0, 1\}$: 1 if the (dedicated) monitor of activity $a$ is on
  duty for the session started at $(r,d,p,h)$. Since each activity has a single
  monitor, the monitor index collapses into the activity index.
* $Run_{a,r,d,p,h} \in \{0, 1\}$: 1 if activity $a$ is active (occupying room $r$)
  at $(d,p,h)$, regardless of when it started.
* $Z \in \mathbb{Z}^+$: auxiliary variable, the minimum number of weekly sessions
  across all activities (max-min fairness).

In [5]:
# binary (1 if activity a starts in room r on day d, period p, slot h)
model.X = pe.Var(model.a, model.r, model.dph, within=pe.Binary)

# binary (1 if the dedicated monitor of activity a is on duty for that session)
model.y = pe.Var(model.a, model.r, model.dph, within=pe.Binary)

# auxiliary variable (minimum number of weekly sessions among all activities)
model.Z = pe.Var(within=pe.NonNegativeIntegers)

# binary (1 if activity a is running in room r on day d, period p, slot h)
model.Run = pe.Var(model.a, model.r, model.dph, within=pe.Binary)

### Objective Function
1. Max-Min Fairness: first we maximize the number of sessions of the *least*
scheduled activity. By maximizing $Z$ we force a varied programme where every
activity gets a fair number of weekly sessions (up to its own target $T_a$).

$$\max Z$$

In [6]:
def obj_rule(m):
    return m.Z

model.obj = pe.Objective(rule=obj_rule, sense=pe.maximize)

2. Total Activity Volume: after fixing the optimal fairness value $Z^*$, the first
objective is deactivated and the second is activated. The model now maximizes the
total number of sessions held (each activity capped at its target $T_a$), keeping
the fairness level guaranteed in step 1.

$$\max \sum_{a \in A} \sum_{r \in R} \sum_{(d,p,h) \in \Omega} X_{a,r,d,p,h}$$

In [7]:
# second objective function: total number of scheduled sessions (activities held)
def total_sessions_rule(m):
    return sum(m.X[a, r, (d, p, h)] for a in m.a for r in m.r for (d, p, h) in m.dph)

model.obj_total = pe.Objective(rule=total_sessions_rule, sense=pe.maximize)
model.obj_total.deactivate()

### Constraints

 1. Valid start time: an activity may only start if the current slot plus its
duration does not go past the end of the period.

$$h + Dur_{a} - 1 \leq \max(H_{p}) \quad \forall a \in A, p \in P, h \in H_{p}$$

In [8]:
# valid start time slots
def valid_start(m, a, p, h):
    return h + pe.value(m.dur[a]) - 1 <= max(m.H[p])

 2. Invalid Start Prohibition: forbid starting an activity in a slot that does not
allow for its full completion within the same period.

$$X_{a,r,d,p,h} = 0 \quad \forall (a,r,d,p,h) \notin \text{ValidStarts}$$

In [9]:
# invalid starts forced to zero
def invalid_start_rule(m, a, r, d, p, h):
    if valid_start(m, a, p, h):
        return pe.Constraint.Skip
    return m.X[a, r, (d, p, h)] == 0

model.invalid_start = pe.Constraint(model.a, model.r, model.dph, rule=invalid_start_rule)

 3. Activity Running Link: links the start variable $X$ with the state variable
$Run$. If activity $a$ starts in room $r$ at slot $t$, it occupies the room for its
whole duration $Dur_a$.

$$Run_{a,r,d,p,h} = \sum_{t \in H_p : t \leq h \leq t + Dur_a - 1} X_{a,r,d,p,t} \quad \forall a, r, (d,p,h)$$

In [10]:
# activity running definition
def running_link_rule(m, a, r, d, p, h):
    return m.Run[a, r, (d, p, h)] == sum(
        m.X[a, r, (d, p, t)]
        for t in m.H[p]
        if valid_start(m, a, p, t) and t <= h <= t + pe.value(m.dur[a]) - 1
    )

model.running_link = pe.Constraint(model.a, model.r, model.dph, rule=running_link_rule)

4. Physical Non-Overlap: ensures no room is double-booked.
   $$\sum_{a \in A} Run_{a,r,d,p,h} \leq 1 \quad \forall r, (d,p,h)$$

In [11]:
# no overlapping activities in the same room
def no_overlap_rule(m, r, d, p, h):
    return sum(m.Run[a, r, (d, p, h)] for a in m.a) <= 1

model.no_overlap = pe.Constraint(model.r, model.dph, rule=no_overlap_rule)

5. Room Availability: activities only occur while the room is open ($RA_{r,d,p,h}$).
   $$\sum_{a \in A} Run_{a,r,d,p,h} \leq RA_{r,d,p,h} \quad \forall r, (d,p,h)$$

In [12]:
# room availability
def room_availability_rule(m, r, d, p, h):
    return sum(m.Run[a, r, (d, p, h)] for a in m.a) <= m.room_avail[r, d, p, h]

model.room_availability = pe.Constraint(model.r, model.dph, rule=room_availability_rule)

6. Mandatory Supervision Requirement: any session that begins must have its
monitor on duty, and the monitor can only be on duty for an actual session.

$$y_{a,r,d,p,h} \geq NS_{a} \cdot X_{a,r,d,p,h} \qquad y_{a,r,d,p,h} \leq X_{a,r,d,p,h} \quad \forall a, r, (d,p,h)$$

In [13]:
# supervisor required (and y only active when the session runs)
def supervisor_required_rule(m, a, r, d, p, h):
    return m.y[a, r, (d, p, h)] >= m.needs_sup[a] * m.X[a, r, (d, p, h)]

model.supervisor_required = pe.Constraint(model.a, model.r, model.dph, rule=supervisor_required_rule)

def supervisor_link_rule(m, a, r, d, p, h):
    return m.y[a, r, (d, p, h)] <= m.X[a, r, (d, p, h)]

model.supervisor_link = pe.Constraint(model.a, model.r, model.dph, rule=supervisor_link_rule)

7. Monitor Availability for Full Duration: a monitor can only be assigned to a
session if available during every slot $k$ of its duration. We precompute
$MA_{a,d,p,h} = \prod_{k=h}^{h+Dur_a-1} SA_{monitor(a),d,p,k}$.

$$y_{a,r,d,p,h} \leq MA_{a,d,p,h} \quad \forall a, r, (d,p,h)$$

In [14]:
# monitor availability across the full duration (precomputed window)
mon_ok = {}
for a in model.a:
    for (d, p, h) in model.dph:
        if not valid_start(model, a, p, h):
            mon_ok[(a, d, p, h)] = 0
            continue
        dur_a = int(pe.value(model.dur[a]))
        s = monitor_of[a]
        mon_ok[(a, d, p, h)] = 1 if all(
            pe.value(model.sup_avail[s, d, p, k]) == 1 for k in range(h, h + dur_a)
        ) else 0

def sup_avail_full_rule(m, a, r, d, p, h):
    return m.y[a, r, (d, p, h)] <= mon_ok[(a, d, p, h)]

model.sup_avail_full = pe.Constraint(model.a, model.r, model.dph, rule=sup_avail_full_rule)

8. Monitor Non-Overlapping: a monitor cannot supervise two active sessions at the
same time (across all rooms). Since each monitor serves a single activity, this also
prevents that activity from running in two rooms at once.

$$\sum_{a : monitor(a)=s} \sum_{r \in R} \sum_{t : t \leq h \leq t + Dur_a - 1} y_{a,r,d,p,t} \leq 1 \quad \forall s, (d,p,h)$$

In [15]:
# monitor no overlap (over its activity and all rooms)
def supervisor_no_overlap_rule(m, s, d, p, h):
    return sum(
        m.y[a, r, (d, p, t)]
        for a in m.a if monitor_of[a] == s
        for r in m.r
        for t in m.H[p]
        if valid_start(m, a, p, t) and t <= h <= t + pe.value(m.dur[a]) - 1
    ) <= 1

model.supervisor_no_overlap = pe.Constraint(model.s, model.dph, rule=supervisor_no_overlap_rule)

9. Room Suitability: an activity can only start in a room that provides every
feature it requires (mirrors for dance, the quiet room for concentration, EXTERIOR
for external activities, ...). A room with a special feature may still host any
generic activity that does not require it.

$$X_{a,r,d,p,h} \leq Su_{a,r} \quad \forall a, r, (d,p,h)$$

In [16]:
# room suitability
def suitability_rule(m, a, r, d, p, h):
    return m.X[a, r, (d, p, h)] <= m.suitable[a, r]

model.suitability = pe.Constraint(model.a, model.r, model.dph, rule=suitability_rule)

10. Noise / Adjacency: if a *musical* activity is actually running in a room, no
*silence* activity may run in an adjacent room at the same time. The constraint is
event-based: if the neighbouring room is hosting a non-musical activity (e.g. yoga
in the gym), a concentration activity next door is perfectly allowed.

$$\sum_{a \in Music} Run_{a,r,d,p,h} + \sum_{a' \in Silence} Run_{a',r',d,p,h} \leq 1 \quad \forall (r,r') : Adj_{r,r'}=1, (d,p,h)$$

In [17]:
# noise / adjacency constraint
model.noise = pe.ConstraintList()

for (d, p, h) in model.dph:
    for r1 in model.r:
        for r2 in model.r:
            if pe.value(model.adjacent[r1, r2]) != 1:
                continue
            model.noise.add(
                sum(model.Run[a, r1, (d, p, h)] for a in model.a if pe.value(model.is_music[a]) == 1)
                + sum(model.Run[a, r2, (d, p, h)] for a in model.a if pe.value(model.is_silence[a]) == 1)
                <= 1
            )

11. No Simultaneous Duplicates: the same activity cannot run in two different rooms
in the same time slot (a single monitor cannot be in two places anyway).

$$\sum_{r \in R} X_{a,r,d,p,h} \leq 1 \quad \forall a, (d,p,h)$$

In [18]:
# an activity is not duplicated across rooms in the same slot
def no_duplicate_rule(m, a, d, p, h):
    return sum(m.X[a, r, (d, p, h)] for r in m.r) <= 1

model.no_duplicate = pe.Constraint(model.a, model.dph, rule=no_duplicate_rule)

12. Session Target (maximum): caps the total number of sessions of each activity at
its weekly target $T_a$. This is the only frequency bound; there is no minimum, so an
activity is held fewer times (even zero) if it cannot be fitted in.

$$\sum_{r \in R} \sum_{(d,p,h) \in \Omega_{valid}} X_{a,r,d,p,h} \leq T_a \quad \forall a \in A$$

In [19]:
# session target: upper bound on weekly sessions per activity
def target_sessions_rule(m, a):
    return sum(
        m.X[a, r, (d, p, h)]
        for r in m.r for (d, p, h) in m.dph
        if valid_start(m, a, p, h)
    ) <= m.target_sessions[a]

model.target_sessions_constraint = pe.Constraint(model.a, rule=target_sessions_rule)

13. Fairness Logic (Max-Min): the auxiliary variable $Z$ is bounded by the number
of weekly sessions of every activity. Maximizing $Z$ pushes up the least-scheduled
activity (bounded above by the smallest target).

$$Z \leq \sum_{r \in R} \sum_{(d,p,h) \in \Omega_{valid}} X_{a,r,d,p,h} \quad \forall a \in A$$

In [20]:
# z definition: minimum number of sessions across all activities
def z_rule(m, a):
    return m.Z <= sum(
        m.X[a, r, (d, p, h)]
        for r in m.r for (d, p, h) in m.dph
        if valid_start(m, a, p, h)
    )

model.z_constraint = pe.Constraint(model.a, rule=z_rule)

#### Solve model

In [21]:
solver = pe.SolverFactory('gurobi')

# first objective function (max-min fairness on number of sessions)
results_1 = solver.solve(model, tee=True)

print("===1===")
print("Solver Status:", results_1.solver.status)
print("Termination Condition:", results_1.solver.termination_condition)

z_star = pe.value(model.Z)

# lock the fairness level reached in phase 1, then switch to the volume objective
model.fairness_lock = pe.Constraint(expr=model.Z >= z_star)

model.obj.deactivate()
model.obj_total.activate()

results_2 = solver.solve(model, tee=True)

print("\n\n===2==")
print("Solver Status:", results_2.solver.status)
print("Termination Condition:", results_2.solver.termination_condition)

Set parameter Username
Set parameter LicenseID to value 2707518
Academic license - for non-commercial use only - expires 2026-09-11
Read LP format model from file C:\Users\Guill\AppData\Local\Temp\tmpeq8xnylb.pyomo.lp
Reading time = 0.60 seconds
x1: 169902 rows, 91855 columns, 506169 nonzeros
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1260P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 169902 rows, 91855 columns and 506169 nonzeros
Model fingerprint: 0xa2cc7301
Variable types: 0 continuous, 91855 integer (91854 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]
Found heuristic solution: objective -0.0000000
Presolve removed 149456 rows and 68568 columns
Presolve time: 0.16s
Presolved: 20446 rows, 23287 

In [22]:
def total_sessions():
    return sum(pe.value(model.X[a, r, (d, p, h)])
               for a in model.a for r in model.r for (d, p, h) in model.dph)

## first objective function
print("First objective function")
print("Z = minimum sessions per activity =", z_star)

## second objective function
print("\nSecond objective function")
print("Z =", pe.value(model.Z))
print("Total sessions scheduled =", round(total_sessions()))

print("\nSessions per activity (scheduled / target)")
for a in model.a:
    cnt = sum(pe.value(model.X[a, r, (d, p, h)]) for r in model.r for (d, p, h) in model.dph)
    print(f"  {a:<18}: {round(cnt)} / {int(pe.value(model.target_sessions[a]))}")

First objective function
Z = minimum sessions per activity = 1.0

Second objective function
Z = 1.0
Total sessions scheduled = 40

Sessions per activity (scheduled / target)
  Yoga              : 5 / 5
  Pilates           : 3 / 3
  Gimnasia          : 3 / 3
  Estiramientos     : 1 / 1
  Hipopresivos      : 1 / 1
  Capoeira          : 1 / 1
  Baile             : 2 / 2
  DanzaVientre      : 1 / 1
  Teatro            : 1 / 1
  CoroMessaDiVoce   : 1 / 1
  CoroGospel        : 1 / 1
  Pintura           : 1 / 1
  Ingles            : 1 / 1
  InformaticaBasica : 1 / 1
  InformaticaMedia  : 1 / 1
  UsoMovilBasico    : 1 / 1
  UsoMovilMedio     : 1 / 1
  MenteActiva       : 2 / 2
  EscrituraLectura  : 1 / 1
  Ortografia        : 1 / 1
  ClubLectura       : 1 / 1
  HistoriasDeVida   : 2 / 2
  AsocRafiki        : 1 / 1
  ReunionAV         : 1 / 1
  ReunionComunidad  : 2 / 2
  Voleibol          : 2 / 2
  GrupoScout        : 1 / 1


In [23]:
print(f"{'DAY':<10} | {'TIME':<6} | {'ROOM':<9} | {'ACTIVITY':<18} | {'MONITOR'}")
print("-" * 70)

# We iterate through the time horizon, rooms and activities
for d in model.d:
    for p in model.p:
        for h in model.H[p]:
            for r in model.r:
                for a in model.a:
                    # We check the 'X' variable (Start Indicator)
                    if pe.value(model.X[a, r, (d, p, h)]) > 0.5:
                        # Monitor on duty for this specific session
                        on_duty = monitor_of[a] if pe.value(model.y[a, r, (d, p, h)]) > 0.5 else "-"
                        print(f"{d:<10} | {slot_label(p, h):<6} | {r:<9} | {a:<18} | {on_duty}")

DAY        | TIME   | ROOM      | ACTIVITY           | MONITOR
----------------------------------------------------------------------
Monday     | 10:00  | Sala7     | Pintura            | Mon_Pintura
Monday     | 10:30  | ZonaOcio  | Gimnasia           | Mon_Gimnasia
Monday     | 11:00  | EXTERIOR  | Voleibol           | -
Monday     | 12:00  | Gimnasio  | Yoga               | Mon_Yoga
Monday     | 16:00  | EXTERIOR  | Voleibol           | -
Monday     | 17:00  | ZonaOcio  | Hipopresivos       | Mon_Hipopresivos
Monday     | 18:00  | ZonaOcio  | AsocRafiki         | Mon_AsocRafiki
Monday     | 18:30  | Sala2     | Teatro             | Mon_Teatro
Monday     | 18:30  | Sala4     | Baile              | Mon_Baile
Tuesday    | 12:00  | Sala2     | Yoga               | Mon_Yoga
Tuesday    | 16:00  | Sala2     | Yoga               | Mon_Yoga
Tuesday    | 17:30  | Sala2     | Gimnasia           | Mon_Gimnasia
Tuesday    | 17:30  | Sala5     | UsoMovilMedio      | Mon_UsoMovilMedio
Tuesday    